In [ ]:
#dependencies
!pip install -q pandas numpy tqdm pyarrow fastparquet
!pip install -q datasets

In [ ]:
#Imports

import csv
import json
import re
from pathlib import Path
from collections import Counter
from datetime import datetime
from pprint import pprint

import numpy as np
import pandas as pd
from tqdm import tqdm

RAW_DIR = Path("/content/sample_data")
CLEAN_DIR = RAW_DIR / "cleaned"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

QUESTION_CANDIDATES = ["question", "query", "prompt", "user", "patient", "instruction", "input"]
ANSWER_CANDIDATES = ["answer", "response", "assistant", "doctor", "output", "completion", "reply"]
LABEL_CANDIDATES = ["label", "category", "intent", "class", "type", "disease", "symptom", "medication", "drug"]
TEXT_CANDIDATES = ["text", "content", "message", "instruction", "input", "output", "question", "answer", "response"]
DISCLAIMER_HINTS = [
    "consult a doctor",
    "consult your doctor",
    "healthcare professional",
    "medical advice",
    "seek medical attention",
    "call emergency",
    "call 911",
    "not a substitute",
]
PLACEHOLDER_TEXTS = {
    "", "na", "n/a", "none", "null", "nan", "test", "placeholder", "lorem ipsum", "asdf", "unknown"
}

print(f"Raw data directory: {RAW_DIR}")
print(f"Clean output directory: {CLEAN_DIR}")


Raw data directory: /content/sample_data
Clean output directory: /content/sample_data/cleaned


In [ ]:
#File discovery

def safe_header(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def list_files(base_dir=RAW_DIR):
    """List supported dataset files under the raw data directory."""
    supported_exts = {".json", ".jsonl", ".csv", ".tsv", ".parquet", ".txt"}
    files = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file() and CLEAN_DIR not in path.parents and path.suffix.lower() in supported_exts:
            files.append(path)
    return files

def detect_file_type(path: Path):
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        return "jsonl"
    if suffix == ".json":
        return "json"
    if suffix == ".csv":
        return "csv"
    if suffix == ".tsv":
        return "tsv"
    if suffix == ".parquet":
        return "parquet"
    if suffix == ".txt":
        return "txt"
    return "unknown"

def read_text_with_fallback(path: Path):
    """Read text with a few common encoding fallbacks."""
    encodings = ["utf-8-sig", "utf-8", "latin-1"]
    last_error = None
    for enc in encodings:
        try:
            return path.read_text(encoding=enc), enc, None
        except Exception as exc:
            last_error = exc
    return None, None, last_error

def sniff_delimiter(text: str, default=","):
    """Guess delimiter for delimited text files."""
    sample = "\n".join(text.splitlines()[:20])
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
        return dialect.delimiter
    except Exception:
        return default

def split_text_records(text: str):
    """Split text files into paragraphs if possible, otherwise non-empty lines."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paragraphs) >= 3:
        return [{"text": p, "source_position": i + 1} for i, p in enumerate(paragraphs)]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return [{"text": line, "source_position": i + 1} for i, line in enumerate(lines)]

def extract_json_records(payload):
    """Turn JSON payload into a record list while preserving nested fields."""
    container_key = None

    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        preferred_keys = ["records", "data", "items", "examples", "samples", "conversations", "messages"]
        records = None
        for key in preferred_keys:
            if key in payload and isinstance(payload[key], list):
                records = payload[key]
                container_key = key
                break
        if records is None:
            records = [payload]
    else:
        records = [{"value": payload}]

    normalized_records = []
    for item in records:
        if isinstance(item, dict):
            normalized_records.append(item)
        else:
            normalized_records.append({"value": item})

    return normalized_records, container_key

def records_to_dataframe(records):
    """Convert records to a DataFrame without flattening nested dict/list values."""
    if not records:
        return pd.DataFrame()
    rows = []
    for rec in records:
        if isinstance(rec, dict):
            rows.append(rec)
        else:
            rows.append({"value": rec})
    return pd.DataFrame(rows)

def load_dataset(path: Path):
    """Load one dataset safely and return metadata plus a DataFrame."""
    file_type = detect_file_type(path)
    logs = []
    malformed_count = 0
    df = pd.DataFrame()
    container_key = None
    encoding = None
    delimiter = None

    try:
        if file_type in {"csv", "tsv"}:
            text, encoding, err = read_text_with_fallback(path)
            if err is not None or text is None:
                raise err if err else ValueError("Unable to read delimited file")
            delimiter = "\t" if file_type == "tsv" else sniff_delimiter(text, default=",")
            df = pd.read_csv(
                path,
                sep=delimiter,
                dtype=str,
                keep_default_na=False,
                na_filter=False,
                engine="python",
                encoding=encoding,
                on_bad_lines="skip",
            )

        elif file_type == "jsonl":
            text, encoding, err = read_text_with_fallback(path)
            if err is not None or text is None:
                raise err if err else ValueError("Unable to read JSONL file")

            records = []
            for line_no, line in enumerate(text.splitlines(), start=1):
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict):
                        records.append(obj)
                    else:
                        records.append({"value": obj})
                except Exception:
                    malformed_count += 1
                    logs.append(f"Malformed JSONL line skipped at line {line_no}")
            df = records_to_dataframe(records)

        elif file_type == "json":
            text, encoding, err = read_text_with_fallback(path)
            if err is not None or text is None:
                raise err if err else ValueError("Unable to read JSON file")

            try:
                payload = json.loads(text)
            except Exception as exc:
                malformed_count += 1
                logs.append(f"Malformed JSON file: {exc}")
                payload = []

            records, container_key = extract_json_records(payload)
            df = records_to_dataframe(records)
        elif file_type == "parquet":
            df = pd.read_parquet(path)
            encoding = "binary/parquet"
        elif file_type == "txt":
            text, encoding, err = read_text_with_fallback(path)
            if err is not None or text is None:
                raise err if err else ValueError("Unable to read text file")
            records = split_text_records(text)
            df = records_to_dataframe(records)

        else:
            logs.append("Unsupported file type")
            df = pd.DataFrame()

    except Exception as exc:
        logs.append(f"Load failed: {exc}")
        df = pd.DataFrame()

    return {
        "path": path,
        "name": path.name,
        "file_type": file_type,
        "df": df,
        "logs": logs,
        "malformed_count": malformed_count,
        "encoding": encoding,
        "delimiter": delimiter,
        "container_key": container_key,
        "original_count": len(df),
    }


In [ ]:
#Inspection, schema inference, and text normalization helpers

def normalize_quotes(text: str):
    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u00b4": "'",
        "`": "'",
    }
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    return text

def clean_text(text):
    """Clean text while preserving meaning and original case."""
    if not isinstance(text, str):
        return text
    text = text.replace("\x00", "")
    text = re.sub(r"[\u00A0\u2000-\u200B\u202F\u205F\u3000]", " ", text)
    text = normalize_quotes(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_nested_value(value):
    """Recursively clean strings inside nested dict/list structures."""
    if isinstance(value, str):
        return clean_text(value)
    if isinstance(value, list):
        return [clean_nested_value(v) for v in value]
    if isinstance(value, dict):
        return {k: clean_nested_value(v) for k, v in value.items()}
    return value

def normalize_column_name(name):
    name = clean_text(str(name))
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unnamed_column"

def normalize_column_names(df: pd.DataFrame):
    """Normalize column names and make duplicates unique."""
    seen = Counter()
    new_cols = []
    for col in df.columns:
        base = normalize_column_name(col)
        seen[base] += 1
        if seen[base] == 1:
            new_cols.append(base)
        else:
            new_cols.append(f"{base}_{seen[base]}")
    out = df.copy()
    out.columns = new_cols
    return out

def split_tokens(name: str):
    return set(str(name).lower().split("_"))

def find_candidate_col(columns, candidates):
    """Find the best matching column by exact token-style matching first."""
    columns = list(columns)

    for cand in candidates:
        for col in columns:
            if col == cand:
                return col

    for cand in candidates:
        for col in columns:
            tokens = split_tokens(col)
            if cand in tokens or col.startswith(cand + "_") or col.endswith("_" + cand):
                return col

    for cand in candidates:
        for col in columns:
            if cand in col:
                return col

    return None

def extract_text_fragments(value):
    """Extract text fragments from scalar or nested structures."""
    fragments = []

    if isinstance(value, str):
        cleaned = clean_text(value)
        if cleaned:
            fragments.append(cleaned)

    elif isinstance(value, list):
        for item in value:
            fragments.extend(extract_text_fragments(item))

    elif isinstance(value, dict):
        for v in value.values():
            fragments.extend(extract_text_fragments(v))

    return fragments

def stringify_value(value):
    if isinstance(value, (dict, list)):
        fragments = extract_text_fragments(value)
        if fragments:
            return " | ".join(fragments)
        try:
            return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
        except Exception:
            return str(value)
    if isinstance(value, str):
        return clean_text(value)
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    return str(value)

def is_blank_value(value):
    if value is None:
        return True
    if isinstance(value, float) and np.isnan(value):
        return True
    if isinstance(value, str):
        return clean_text(value) == ""
    if isinstance(value, list):
        return len(extract_text_fragments(value)) == 0 and len(value) == 0
    if isinstance(value, dict):
        return all(is_blank_value(v) for v in value.values()) if value else True
    return False

def is_meaningful_text(text, min_alpha=2):
    if not isinstance(text, str):
        return False
    text = clean_text(text)
    if not text:
        return False
    alpha_count = sum(ch.isalpha() for ch in text)
    return alpha_count >= min_alpha or len(text) >= 8

def normalized_match_text(text):
    text = clean_text(text if isinstance(text, str) else stringify_value(text))
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]+", "", text, flags=re.UNICODE)
    return text.strip()

def safe_row_signature(row_dict):
    def normalize_for_signature(value):
        if isinstance(value, str):
            return clean_text(value)
        if isinstance(value, list):
            return [normalize_for_signature(v) for v in value]
        if isinstance(value, dict):
            return {str(k): normalize_for_signature(v) for k, v in sorted(value.items(), key=lambda x: str(x[0]))}
        if value is None:
            return None
        if isinstance(value, float) and np.isnan(value):
            return None
        return value

    normalized = {str(k): normalize_for_signature(v) for k, v in sorted(row_dict.items(), key=lambda x: str(x[0]))}
    return json.dumps(normalized, ensure_ascii=False, sort_keys=True, default=str)

def infer_schema(df: pd.DataFrame):
    """Infer a simple schema label for routing cleaning logic."""
    cols = list(df.columns)
    q_col = find_candidate_col(cols, QUESTION_CANDIDATES)
    a_col = find_candidate_col(cols, ANSWER_CANDIDATES)
    label_col = find_candidate_col(cols, LABEL_CANDIDATES)

    if "messages" in cols or "conversation" in cols or "dialogue" in cols:
        return "dialogue"
    if "role" in cols and "content" in cols:
        return "dialogue"
    if "instruction" in cols and ("output" in cols or "response" in cols):
        return "instruction"
    if q_col and a_col and q_col != a_col:
        return "qa"
    if label_col or any(x in cols for x in ["symptom", "disease", "drug", "medication", "treatment"]):
        return "structured_table"

    object_like_cols = [c for c in cols if df[c].dtype == "object"]
    if len(object_like_cols) == 1:
        return "text"
    return "generic_table"

def missing_empty_counts(df: pd.DataFrame):
    counts = {}
    for col in df.columns:
        missing = 0
        for value in df[col].tolist():
            if is_blank_value(value):
                missing += 1
        counts[col] = missing
    return counts

def duplicate_row_count(df: pd.DataFrame):
    if df.empty:
        return 0
    signatures = df.apply(lambda row: safe_row_signature(row.to_dict()), axis=1)
    return int(signatures.duplicated().sum())

def inspect_dataset(load_result):
    """Print inspection details and return a compact summary dict."""
    path = load_result["path"]
    df = load_result["df"]
    file_type = load_result["file_type"]

    safe_header(f"INSPECTING: {path.name}")

    print(f"Path: {path}")
    print(f"Type: {file_type}")
    print(f"Encoding: {load_result['encoding']}")
    if load_result["delimiter"] is not None:
        print(f"Detected delimiter: {repr(load_result['delimiter'])}")
    if load_result["container_key"] is not None:
        print(f"JSON container key: {load_result['container_key']}")
    print(f"Loaded rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Schema hint: {infer_schema(df) if not df.empty else 'unknown'}")

    if load_result["logs"]:
        print("\nLogs:")
        for msg in load_result["logs"][:10]:
            print(f"- {msg}")

    if df.empty:
        print("\nNo rows loaded.")
        return {
            "file": path.name,
            "type": file_type,
            "rows": 0,
            "columns": [],
            "schema": "unknown",
            "duplicate_rows": 0,
        }

    print("\nFirst 5 parsed rows:")
    samples = df.head(5).to_dict(orient="records")
    pprint(samples, width=160, sort_dicts=False)

    print("\nMissing / empty values by column:")
    miss = missing_empty_counts(df)
    print(pd.Series(miss).sort_values(ascending=False).to_string())

    dup_count = duplicate_row_count(df)
    print(f"\nDuplicate rows detected: {dup_count}")

    text_cols = [c for c in df.columns if df[c].dtype == "object"]
    if text_cols:
        print("\nText length stats:")
        text_stats = []
        for col in text_cols:
            lengths = []
            for value in df[col].tolist():
                text_value = stringify_value(value)
                if text_value:
                    lengths.append(len(text_value))
            if lengths:
                text_stats.append({
                    "column": col,
                    "count": len(lengths),
                    "min": int(np.min(lengths)),
                    "median": float(np.median(lengths)),
                    "mean": float(np.mean(lengths)),
                    "max": int(np.max(lengths)),
                })
        if text_stats:
            print(pd.DataFrame(text_stats).to_string(index=False))
        else:
            print("No non-empty text content found.")

    key_cols = []
    for candidate_group in [LABEL_CANDIDATES, ["role"], ["speaker"], ["source"], ["dataset"]]:
        col = find_candidate_col(df.columns, candidate_group)
        if col and col not in key_cols:
            key_cols.append(col)

    if key_cols:
        print("\nUnique values for key columns:")
        for col in key_cols:
            value_counts = df[col].astype(str).value_counts(dropna=False).head(10)
            print(f"\nColumn: {col}")
            print(value_counts.to_string())

    return {
        "file": path.name,
        "type": file_type,
        "rows": len(df),
        "columns": list(df.columns),
        "schema": infer_schema(df),
        "duplicate_rows": dup_count,
    }


In [ ]:
#Cleaning, deduplication, quarantine, saving, and reporting helpers

def clean_dataframe_values(df: pd.DataFrame):
    """Clean all cells and convert empty strings to NaN."""
    out = df.copy()

    def clean_cell(value):
        value = clean_nested_value(value)
        if isinstance(value, str):
            value = clean_text(value)
            return np.nan if value == "" else value
        return value

    for col in out.columns:
        out[col] = out[col].map(clean_cell)

    return out

def validate_record(record, schema_hint):
    """Basic structural validation without rewriting meaning."""
    reasons = []

    if not record or all(is_blank_value(v) for v in record.values()):
        reasons.append("fully_empty")
        return False, reasons

    cols = list(record.keys())
    q_col = find_candidate_col(cols, QUESTION_CANDIDATES)
    a_col = find_candidate_col(cols, ANSWER_CANDIDATES)

    if schema_hint == "dialogue":
        if "messages" in record:
            messages = record.get("messages")
            if not isinstance(messages, list) or len(messages) < 2:
                reasons.append("invalid_messages_structure")
            else:
                valid_messages = 0
                seen_roles = set()
                for msg in messages:
                    if isinstance(msg, dict):
                        role = clean_text(str(msg.get("role", ""))).lower()
                        content = msg.get("content", "")
                        if role and extract_text_fragments(content):
                            valid_messages += 1
                            seen_roles.add(role)
                if valid_messages < 2:
                    reasons.append("too_few_valid_messages")
                if not any(r in seen_roles for r in {"user", "patient", "human"}):
                    reasons.append("missing_user_role")
                if not any(r in seen_roles for r in {"assistant", "doctor", "bot"}):
                    reasons.append("missing_assistant_role")
        elif q_col and a_col:
            q = stringify_value(record.get(q_col))
            a = stringify_value(record.get(a_col))
            if not is_meaningful_text(q):
                reasons.append("missing_user_content")
            if not is_meaningful_text(a):
                reasons.append("missing_assistant_content")
        elif "role" in record and "content" in record:
            if not is_meaningful_text(stringify_value(record.get("content"))):
                reasons.append("empty_dialogue_content")
        else:
            reasons.append("unknown_dialogue_layout")

    elif schema_hint in {"qa", "instruction"}:
        if q_col and not is_meaningful_text(stringify_value(record.get(q_col)), min_alpha=2):
            reasons.append("missing_or_short_prompt")
        if a_col and not is_meaningful_text(stringify_value(record.get(a_col)), min_alpha=2):
            reasons.append("missing_or_short_answer")
        if not q_col and not a_col:
            reasons.append("missing_qa_fields")

    elif schema_hint == "text":
        combined = " ".join(extract_text_fragments(record))
        if not is_meaningful_text(combined, min_alpha=2):
            reasons.append("empty_text")

    else:
        non_blank_fields = sum(not is_blank_value(v) for v in record.values())
        if non_blank_fields == 0:
            reasons.append("fully_empty")

    return len(reasons) == 0, reasons

def detect_dominant_boilerplate(df: pd.DataFrame):
    """Detect repeated disclaimer-like answers that dominate a dataset."""
    if df.empty:
        return set()

    cols = list(df.columns)
    answer_col = find_candidate_col(cols, ANSWER_CANDIDATES)
    candidate_cols = [answer_col] if answer_col else [c for c in cols if df[c].dtype == "object"]

    boilerplate_hits = set()

    for col in candidate_cols:
        if col is None:
            continue
        normalized_values = []
        for value in df[col].tolist():
            text = stringify_value(value)
            if len(text) >= 25:
                normalized_values.append(normalized_match_text(text))

        if len(normalized_values) < 5:
            continue

        counts = Counter(normalized_values)
        top_text, top_count = counts.most_common(1)[0]
        dominance_ratio = top_count / max(len(normalized_values), 1)

        if dominance_ratio >= 0.35 and any(hint.replace(" ", "") in top_text.replace(" ", "") for hint in DISCLAIMER_HINTS):
            boilerplate_hits.add(top_text)

    return boilerplate_hits

def quarantine_record(record, schema_hint, dominant_boilerplate=None):
    """Flag suspicious rows into quarantine instead of silently dropping them."""
    reasons = []
    dominant_boilerplate = dominant_boilerplate or set()

    combined_text = " ".join(extract_text_fragments(record))
    combined_norm = normalized_match_text(combined_text)

    if combined_norm in PLACEHOLDER_TEXTS:
        reasons.append("placeholder_text")

    if any(marker in combined_text for marker in ["�", "Ã", "Â", "â€"]):
        reasons.append("encoding_corruption")

    if re.search(r"(.)\1{10,}", combined_text):
        reasons.append("repetitive_characters")

    if re.search(r"ignore previous instructions|system prompt|jailbreak|as an ai language model", combined_text, flags=re.I):
        reasons.append("meta_or_prompt_injection_text")

    visible_chars = [ch for ch in combined_text if not ch.isspace()]
    alpha_chars = [ch for ch in combined_text if ch.isalpha()]
    if len(visible_chars) >= 12:
        symbol_ratio = 1 - (len(alpha_chars) / max(len(visible_chars), 1))
        if symbol_ratio > 0.75:
            reasons.append("mostly_symbols_or_noise")

    if schema_hint in {"qa", "instruction", "dialogue"} and len(alpha_chars) < 3:
        reasons.append("too_short_for_supervised_medical_text")

    cols = list(record.keys())
    answer_col = find_candidate_col(cols, ANSWER_CANDIDATES)
    if answer_col:
        ans_norm = normalized_match_text(stringify_value(record.get(answer_col)))
        if ans_norm in dominant_boilerplate:
            reasons.append("dominant_boilerplate_disclaimer")

    return len(reasons) > 0, sorted(set(reasons))

def deduplicate_records(df: pd.DataFrame, schema_hint: str):
    """Remove exact and normalized text duplicates."""
    if df.empty:
        return df.copy(), {"exact_row_duplicates": 0, "normalized_text_duplicates": 0, "qa_duplicates": 0}

    work = df.copy().reset_index(drop=True)
    removed = {"exact_row_duplicates": 0, "normalized_text_duplicates": 0, "qa_duplicates": 0}

    row_keys = work.apply(lambda row: safe_row_signature(row.to_dict()), axis=1)
    exact_mask = row_keys.duplicated()
    removed["exact_row_duplicates"] = int(exact_mask.sum())
    work = work.loc[~exact_mask].copy().reset_index(drop=True)

    q_col = find_candidate_col(work.columns, QUESTION_CANDIDATES)
    a_col = find_candidate_col(work.columns, ANSWER_CANDIDATES)

    if q_col and a_col and q_col != a_col:
        qa_keys = (
            work[q_col].map(lambda v: normalized_match_text(stringify_value(v))) + " ||| " +
            work[a_col].map(lambda v: normalized_match_text(stringify_value(v)))
        )
        qa_mask = qa_keys.duplicated()
        removed["qa_duplicates"] = int(qa_mask.sum())
        work = work.loc[~qa_mask].copy().reset_index(drop=True)
    else:
        text_cols = [c for c in work.columns if work[c].dtype == "object"]
        if text_cols:
            text_keys = work.apply(
                lambda row: " ||| ".join(
                    normalized_match_text(stringify_value(row[col])) for col in text_cols
                ),
                axis=1,
            )
            text_mask = text_keys.duplicated()
            removed["normalized_text_duplicates"] = int(text_mask.sum())
            work = work.loc[~text_mask].copy().reset_index(drop=True)

    return work, removed

def collect_top_values(df: pd.DataFrame):
    """Collect a few useful top repeated labels/symptoms/categories when available."""
    result = {}
    for col_name in ["label", "category", "intent", "disease", "symptom", "drug", "medication", "type"]:
        if col_name in df.columns:
            counts = df[col_name].dropna().astype(str).value_counts().head(10).to_dict()
            if counts:
                result[col_name] = counts
    return result

def clean_dataset(load_result):
    """End-to-end cleaning for one dataset with quarantine tracking."""
    raw_df = load_result["df"].copy()
    file_type = load_result["file_type"]
    source_path = load_result["path"]

    stats = {
        "file": source_path.name,
        "file_type": file_type,
        "original_count": len(raw_df),
        "cleaned_count": 0,
        "removed_malformed_records": int(load_result["malformed_count"]),
        "removed_empty_rows": 0,
        "removed_invalid_records": 0,
        "removed_duplicates": 0,
        "quarantined_records": 0,
        "schema": "unknown",
        "columns": [],
        "top_values": {},
        "logs": list(load_result["logs"]),
    }

    if raw_df.empty:
        stats["logs"].append("Dataset is empty after loading.")
        return {
            "source_path": source_path,
            "file_type": file_type,
            "schema": "unknown",
            "cleaned_df": pd.DataFrame(),
            "quarantine_rows": [],
            "stats": stats,
        }

    df = normalize_column_names(raw_df)
    df = clean_dataframe_values(df)

    fully_empty_mask = df.apply(lambda row: all(is_blank_value(v) for v in row.to_dict().values()), axis=1)
    stats["removed_empty_rows"] = int(fully_empty_mask.sum())
    df = df.loc[~fully_empty_mask].copy().reset_index(drop=True)

    schema_hint = infer_schema(df)
    stats["schema"] = schema_hint
    stats["columns"] = list(df.columns)

    dominant_boilerplate = detect_dominant_boilerplate(df)

    cleaned_rows = []
    quarantine_rows = []

    for _, row in df.iterrows():
        record = row.to_dict()
        is_valid, invalid_reasons = validate_record(record, schema_hint)
        should_quarantine, quarantine_reasons = quarantine_record(record, schema_hint, dominant_boilerplate)

        if not is_valid:
            stats["removed_invalid_records"] += 1
            quarantine_rows.append({
                "source_file": source_path.name,
                "schema": schema_hint,
                "reasons": invalid_reasons,
                "record": record,
            })
            continue

        if should_quarantine:
            quarantine_rows.append({
                "source_file": source_path.name,
                "schema": schema_hint,
                "reasons": quarantine_reasons,
                "record": record,
            })
            continue
        record["_source_dataset"] = source_path.name
        cleaned_rows.append(record)

    cleaned_df = pd.DataFrame(cleaned_rows)
    if not cleaned_df.empty:
        cleaned_df = cleaned_df.reindex(columns=stats["columns"])

    cleaned_df, dup_stats = deduplicate_records(cleaned_df, schema_hint)
    stats["removed_duplicates"] = int(sum(dup_stats.values()))
    stats["cleaned_count"] = int(len(cleaned_df))
    stats["quarantined_records"] = int(len(quarantine_rows))
    stats["top_values"] = collect_top_values(cleaned_df)

    return {
        "source_path": source_path,
        "file_type": file_type,
        "schema": schema_hint,
        "cleaned_df": cleaned_df,
        "quarantine_rows": quarantine_rows,
        "stats": stats,
    }

def make_json_serializable(value):
    if isinstance(value, dict):
        return {str(k): make_json_serializable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [make_json_serializable(v) for v in value]
    if isinstance(value, float) and np.isnan(value):
        return None
    if pd.isna(value) if not isinstance(value, (dict, list, str)) else False:
        return None
    return value

def save_jsonl_records(records, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(make_json_serializable(record), ensure_ascii=False) + "\n")

def save_cleaned(clean_result, output_dir=CLEAN_DIR):
    """Save cleaned dataset in a matching or close matching format."""
    source_path = clean_result["source_path"]
    file_type = clean_result["file_type"]
    cleaned_df = clean_result["cleaned_df"].copy()

    if file_type == "csv":
        out_path = output_dir / source_path.name
        cleaned_df.to_csv(out_path, index=False, encoding="utf-8")
    elif file_type == "tsv":
        out_path = output_dir / source_path.name
        cleaned_df.to_csv(out_path, index=False, sep="\t", encoding="utf-8")
    elif file_type in {"json", "jsonl"}:
        out_path = output_dir / f"{source_path.stem}.jsonl"
        save_jsonl_records(cleaned_df.to_dict(orient="records"), out_path)
    elif file_type == "txt":
        if set(cleaned_df.columns).issubset({"text", "source_position"}):
            out_path = output_dir / source_path.name
            lines = cleaned_df.get("text", pd.Series(dtype=str)).fillna("").astype(str).tolist()
            out_path.write_text("\n".join(lines), encoding="utf-8")
        else:
            out_path = output_dir / f"{source_path.stem}.csv"
            cleaned_df.to_csv(out_path, index=False, encoding="utf-8")
    else:
        out_path = output_dir / f"{source_path.stem}.csv"
        cleaned_df.to_csv(out_path, index=False, encoding="utf-8")

    return out_path

def generate_report(clean_results):
    """Build summary CSV/JSON report objects."""
    rows = []
    column_counter = Counter()

    for result in clean_results:
        stats = result["stats"]
        column_counter.update(stats["columns"])
        rows.append({
            "file": stats["file"],
            "file_type": stats["file_type"],
            "schema": stats["schema"],
            "original_count": stats["original_count"],
            "cleaned_count": stats["cleaned_count"],
            "removed_malformed_records": stats["removed_malformed_records"],
            "removed_empty_rows": stats["removed_empty_rows"],
            "removed_invalid_records": stats["removed_invalid_records"],
            "removed_duplicates": stats["removed_duplicates"],
            "quarantined_records": stats["quarantined_records"],
            "columns": ", ".join(stats["columns"]),
            "top_values": json.dumps(stats["top_values"], ensure_ascii=False),
        })

    summary_df = pd.DataFrame(rows).sort_values(["file"], ascending=True) if rows else pd.DataFrame()

    report_json = {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "raw_dir": str(RAW_DIR),
        "clean_dir": str(CLEAN_DIR),
        "files_processed": len(clean_results),
        "global_totals": {
            "original_count": int(sum(r["stats"]["original_count"] for r in clean_results)),
            "cleaned_count": int(sum(r["stats"]["cleaned_count"] for r in clean_results)),
            "removed_malformed_records": int(sum(r["stats"]["removed_malformed_records"] for r in clean_results)),
            "removed_empty_rows": int(sum(r["stats"]["removed_empty_rows"] for r in clean_results)),
            "removed_invalid_records": int(sum(r["stats"]["removed_invalid_records"] for r in clean_results)),
            "removed_duplicates": int(sum(r["stats"]["removed_duplicates"] for r in clean_results)),
            "quarantined_records": int(sum(r["stats"]["quarantined_records"] for r in clean_results)),
        },
        "most_common_columns": column_counter.most_common(25),
        "per_file": [r["stats"] for r in clean_results],
    }

    return summary_df, report_json


In [ ]:
#Discover files and inspect every raw dataset

raw_files = list_files(RAW_DIR)

safe_header("RAW FILE DISCOVERY")
if not raw_files:
    print("No supported dataset files found in /content/sample_data/")
else:
    for i, path in enumerate(raw_files, start=1):
        print(f"{i:02d}. {path.name}  |  extension={path.suffix.lower()}  |  path={path}")

loaded_datasets = {}
inspection_rows = []

for path in raw_files:
    try:
        loaded = load_dataset(path)
        loaded_datasets[str(path)] = loaded
        inspection_rows.append(inspect_dataset(loaded))
    except Exception as exc:
        print(f"\nFailed during inspection for {path.name}: {exc}")

safe_header("INSPECTION SUMMARY TABLE")
inspection_df = pd.DataFrame(inspection_rows)
if inspection_df.empty:
    print("No inspection summary available.")
else:
    print(inspection_df.to_string(index=False))



RAW FILE DISCOVERY
01. DB_compounds_lipinski.csv  |  extension=.csv  |  path=/content/sample_data/DB_compounds_lipinski.csv
02. GenMedGPT-5k.json  |  extension=.json  |  path=/content/sample_data/GenMedGPT-5k.json
03. HealthCareMagic-100k.json  |  extension=.json  |  path=/content/sample_data/HealthCareMagic-100k.json
04. disease_database_mini.csv  |  extension=.csv  |  path=/content/sample_data/disease_database_mini.csv
05. format_dataset.csv  |  extension=.csv  |  path=/content/sample_data/format_dataset.csv
06. iCliniq.json  |  extension=.json  |  path=/content/sample_data/iCliniq.json
07. indications.tsv  |  extension=.tsv  |  path=/content/sample_data/indications.tsv
08. side-effect-terms.tsv  |  extension=.tsv  |  path=/content/sample_data/side-effect-terms.tsv
09. side-effects.tsv  |  extension=.tsv  |  path=/content/sample_data/side-effects.tsv

INSPECTING: DB_compounds_lipinski.csv
Path: /content/sample_data/DB_compounds_lipinski.csv
Type: csv
Encoding: utf-8-sig
Detected del

In [ ]:
#Clean all datasets, save cleaned outputs, and build quarantine files

import gc

clean_results = []

SKIP_FILES = {
    "DB_compounds_lipinski.csv"
}

all_quarantine_rows = []
saved_clean_paths = []

for path_str, loaded in tqdm(loaded_datasets.items()):

    # Skip irrelevant datasets
    if loaded["name"] in SKIP_FILES:
        print(f"Skipping irrelevant dataset: {loaded['name']}")
        continue

    safe_header(f"CLEANING: {loaded['name']}")

    try:
        # Run cleaning pipeline
        result = clean_dataset(loaded)
        clean_results.append(result)

        # Save cleaned dataset
        out_path = save_cleaned(result, CLEAN_DIR)
        saved_clean_paths.append(out_path)

        # Collect quarantined rows
        for q in result["quarantine_rows"]:
            all_quarantine_rows.append(q)

        # Print cleaning stats
        stats = result["stats"]

        print(f"Saved cleaned file: {out_path}")
        print(f"Schema: {stats['schema']}")
        print(f"Original records: {stats['original_count']}")
        print(f"Cleaned records: {stats['cleaned_count']}")
        print(f"Malformed removed: {stats['removed_malformed_records']}")
        print(f"Empty removed: {stats['removed_empty_rows']}")
        print(f"Invalid removed: {stats['removed_invalid_records']}")
        print(f"Duplicates removed: {stats['removed_duplicates']}")
        print(f"Quarantined: {stats['quarantined_records']}")

        # Print logs if available
        if stats["logs"]:
            print("Logs:")
            for msg in stats["logs"][:10]:
                print(f"- {msg}")

        # Memory cleanup for Colab stability
        del result
        gc.collect()

    except Exception as exc:
        print(f"Cleaning failed for {loaded['name']}: {exc}")

# Save combined quarantine file
quarantine_path = CLEAN_DIR / "quarantine_records.jsonl"
save_jsonl_records(all_quarantine_rows, quarantine_path)

safe_header("CLEANING COMPLETE")

print(f"Cleaned files saved: {len(saved_clean_paths)}")
print(f"Combined quarantine file: {quarantine_path}")
print(f"Quarantined records total: {len(all_quarantine_rows)}")

  0%|          | 0/9 [00:00<?, ?it/s]

Skipping irrelevant dataset: DB_compounds_lipinski.csv

CLEANING: GenMedGPT-5k.json


 22%|██▏       | 2/9 [00:16<00:56,  8.02s/it]

Saved cleaned file: /content/sample_data/cleaned/GenMedGPT-5k.jsonl
Schema: instruction
Original records: 5452
Cleaned records: 5431
Malformed removed: 0
Empty removed: 0
Invalid removed: 0
Duplicates removed: 14
Quarantined: 7

CLEANING: HealthCareMagic-100k.json


In [ ]:
#Generate and save summary reports

summary_df, report_json = generate_report(clean_results)

summary_csv_path = CLEAN_DIR / "cleaning_report_summary.csv"
report_json_path = CLEAN_DIR / "cleaning_report.json"

summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8")
with report_json_path.open("w", encoding="utf-8") as f:
    json.dump(report_json, f, ensure_ascii=False, indent=2)

safe_header("SUMMARY REPORT")
if summary_df.empty:
    print("No summary rows to report.")
else:
    print(summary_df.to_string(index=False))

print(f"\nSaved summary CSV: {summary_csv_path}")
print(f"Saved report JSON: {report_json_path}")


In [ ]:
#Final sanity check: reload cleaned outputs and preview each file

cleaned_files = [
    p for p in sorted(CLEAN_DIR.iterdir())
    if p.is_file() and p.name not in {".DS_Store"}
]

safe_header("SANITY CHECK: RELOADING CLEANED OUTPUTS")

if not cleaned_files:
    print("No cleaned files found.")
else:
    for path in cleaned_files:
        print("\n" + "-" * 100)
        print(f"Previewing: {path.name}")

        try:
            loaded = load_dataset(path)
            df = loaded["df"]

            print(f"Type: {loaded['file_type']}")
            print(f"Rows: {len(df)}")
            print(f"Columns: {list(df.columns)}")

            if not df.empty:
                preview = df.head(5).to_dict(orient="records")
                pprint(preview, width=160, sort_dicts=False)
            else:
                print("File loaded but contains no rows.")

        except Exception as exc:
            print(f"Could not preview {path.name}: {exc}")
